# Week 09: Find, explain, change, preserve

Student exercise: write your own predictions and explanations.

## How to use this notebook

**Monday 2 November 2026, 14:00–17:00.** CPU only. No paid tool, API key, GPU or downloaded dataset. The same file works in free Colab and local Jupyter. In Jupyter select a Python kernel with NumPy, Matplotlib and scikit-learn. Setup installs a missing package only; it does not upgrade an already installed environment.

1. Save your own copy (Colab: File → Save a copy in Drive; Jupyter: save the .ipynb).
2. Run Setup once. Then complete Tasks **1 through 8 in order**. Each task has a prediction field, runnable scaffold and explanation field; you do not need to create cells.
3. H1 uses Tasks 1–2 (15 practice minutes). H2 uses Tasks 3–5 (45). H3 uses Tasks 6–8 (40), then peer explanation/export/optional work (20). Explanation time is 60 minutes; practice is 120.
4. If the kernel resets or you see `NameError`, run from the top. Keep the same notebook through H2 and H3. The worked notebook is an alternative with example responses, not a prerequisite that you must run first.
5. Save this edited notebook and download `week09-evidence.zip` before the temporary runtime ends. Colab: click the folder icon, then the three-dot menu beside the ZIP → Download. Jupyter: follow the file link at the end.

**Evidence boundary:** independent synthetic rows, dimensionless inputs, synthetic response units. These are not real equipment measurements, time-series forecasts, financial claims or deployment validation. A known generator makes the representation error inspectable. Existing written/oral examination and designated Homework policies are unchanged.

**Prerequisites:** predict versus observe; train/validation/test; a line and a square; basic Python arrays. No calculus or bootstrap knowledge is required for the eight core tasks.


## Setup

Run once. This creates only the `week09-artifacts` output directory and clears files from previous runs inside that directory. The source CSV is generated and frozen in Task 3, so Tasks 1–2 stay simple. Package versions are printed and exported. If setup installed packages but an import remains broken, restart the runtime once and run from the top.

In [ ]:

from pathlib import Path
import sys, subprocess, importlib.util
needed={'numpy':'numpy','matplotlib':'matplotlib','sklearn':'scikit-learn'}
missing=[package for module,package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import numpy as np
import matplotlib.pyplot as plt
import sklearn, json, hashlib, csv, io, zipfile, platform
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.base import clone
from IPython.display import display, FileLink
ARTIFACTS=Path('week09-artifacts'); ARTIFACTS.mkdir(exist_ok=True)
# Clear old outputs from this notebook's own output folder, never other folders.
for old in ARTIFACTS.iterdir():
    if old.is_file(): old.unlink()
CONFIG={'seed':2026,'n':500,'noise_sd':0.2,'ridge_alpha':1.0,
        'stress_scales':[1.0,1.5,2.5],'stress_n':150,'slice_boundary':1.0,
        'delta_x0':0.5,'prediction_atol':1e-10,'prediction_rtol':1e-10}
responses={}
VERSIONS={'python':platform.python_version(),'numpy':np.__version__,
          'scikit-learn':sklearn.__version__}
plt.rcParams.update({'figure.figsize':(9,4.8),'font.size':12,'axes.spines.top':False,
                     'axes.spines.right':False,'figure.dpi':120})
COLORS={'linear':'#a43a4b','quadratic':'#147c80'}
def metrics(y_true,pred):
    residual=np.asarray(pred)-np.asarray(y_true)
    return {'MAE':float(np.mean(np.abs(residual))), 'RMSE':float(np.sqrt(np.mean(residual**2)))}
def savefig(name):
    plt.tight_layout(); plt.savefig(ARTIFACTS/name,bbox_inches='tight'); plt.show(); plt.close()
def add_square(a):
    a=np.asarray(a); return np.column_stack([a,a[:,1]**2])
def candidates():
    return {'linear':make_pipeline(StandardScaler(),Ridge(alpha=CONFIG['ridge_alpha'],solver='svd')),
            'quadratic':make_pipeline(FunctionTransformer(add_square,validate=True),StandardScaler(),
                                      Ridge(alpha=CONFIG['ridge_alpha'],solver='svd'))}
def snapshot(model):
    scaler=model.named_steps['standardscaler']; ridge=model.named_steps['ridge']
    return [a.copy() for a in [scaler.mean_,scaler.scale_,ridge.coef_,np.atleast_1d(ridge.intercept_)]]
def unchanged(before,model):
    return all(np.array_equal(a,b) for a,b in zip(before,snapshot(model)))
print('CPU environment:',VERSIONS)
print('No API key, GPU, external dataset or paid AI tool is needed.')


## Task 1: Read five signed errors (8 min)

**Purpose:** Distinguish an observed value, a prediction and a signed residual.

**Do:** Hold x0=0 and noise=0. The synthetic response is x1 squared. We deliberately draw a fixed prediction of 1; it was not fitted to these five points. Predict which rows are underpredicted. Run the cell, check both sums, then explain one negative residual. Stop when you can point to the matching vertical error segment.

**Check / stopping condition:** Residuals [-3,0,1,0,-3]; absolute sum 7; squared sum 19; MAE 1.4; RMSE sqrt(3.8), about 1.949.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task1_prediction"] = ''

In [ ]:
x_demo=np.array([-2.,-1.,0.,1.,2.]); y_demo=x_demo**2
pred_demo=np.ones(5); residual_demo=pred_demo-y_demo
assert np.array_equal(residual_demo,[-3,0,1,0,-3])
warmup=metrics(y_demo,pred_demo)
print('x1 | observed | predicted | residual')
for row in zip(x_demo,y_demo,pred_demo,residual_demo): print(row)
print('Absolute sum:',np.abs(residual_demo).sum(),'Squared sum:',(residual_demo**2).sum(),warmup)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(x_demo,y_demo,label='Observed square',c='#147c80',s=70)
axes[0].plot(x_demo,pred_demo,'o--',label='Fixed prediction = 1',c='#a43a4b')
axes[0].vlines(x_demo,np.minimum(y_demo,pred_demo),np.maximum(y_demo,pred_demo),color='#64748b')
axes[0].set(xlabel='x1 (dimensionless)',ylabel='Synthetic response'); axes[0].legend()
axes[1].axhline(0,color='black'); axes[1].bar(x_demo,residual_demo,color=['#a43a4b' if x<0 else '#147c80' for x in residual_demo])
axes[1].set(xlabel='x1',ylabel='Prediction - observation',ylim=(-4,2))
fig.suptitle('Teaching slice: a declared approximation, not a fitted experiment')
savefig('01-five-errors.png')

In [ ]:
responses["task1_explanation"] = ''

## Task 2: Change the group mixture (7 min)

**Purpose:** Explain why an aggregate score changes even when group errors do not.

**Do:** Use the paper illustration: every easy-case absolute error is 1 and every difficult-case error is 10. Predict the aggregate when difficult cases increase from 10 to 25 of the same 100 cases. Compute each group contribution before adding. These invented group sizes are separate from the actual 75-row test set used below.

**Check / stopping condition:** 90/10 gives MAE 1.9; 75/25 gives MAE 3.25. The group losses are unchanged; the mixture changed.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task2_prediction"] = ''

In [ ]:
mixture=[]
for easy,hard in [(90,10),(75,25)]:
    parts=[easy*1/100,hard*10/100]
    mixture.append({'easy':easy,'hard':hard,'parts':parts,'MAE':sum(parts)})
print(mixture)
assert np.allclose([a['MAE'] for a in mixture],[1.9,3.25])
plt.figure(); bottoms=np.array([r['parts'][0] for r in mixture]); tops=np.array([r['parts'][1] for r in mixture])
plt.bar(['90 / 10','75 / 25'],bottoms,label='Easy contribution',color='#147c80')
plt.bar(['90 / 10','75 / 25'],tops,bottom=bottoms,label='Difficult contribution',color='#a43a4b')
plt.ylabel('Weighted MAE (synthetic response units)'); plt.xlabel('Easy / difficult case counts (100 total)')
plt.title('Paper illustration: same group losses, changed mixture'); plt.legend(); savefig('02-mixture.png')

In [ ]:
responses["task2_explanation"] = ''

## Task 3: Fit, select and freeze (18 min)

**Purpose:** Keep training-only fitting separate from validation selection and final evaluation.

**Do:** Generate the declared 500 independent synthetic rows. Save IDs and 350/75/75 split membership before fitting. x0 supplies linear signal, x1 a square, x2 is a negative control. Fit both pipelines on training rows only. Choose by validation MAE (exact ties favor linear). Save that choice before opening test results in Task 4. Stop when disjoint IDs, scaler training mean and the saved choice all pass.

**Check / stopping condition:** 350 training, 75 validation and 75 test IDs with no overlap. Actual validation scores are printed by your run; do not copy a score from a slide.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task3_prediction"] = ''

In [ ]:
rng=np.random.default_rng(CONFIG['seed']); X=rng.normal(size=(CONFIG['n'],3))
noise=rng.normal(0,CONFIG['noise_sd'],size=CONFIG['n']); y=2*X[:,0]+X[:,1]**2+noise
ids=np.arange(len(X)); tr,held=train_test_split(ids,test_size=.30,random_state=CONFIG['seed'])
va,te=train_test_split(held,test_size=.50,random_state=CONFIG['seed']+1)
tr,va,te=map(np.sort,(tr,va,te)); assert tuple(map(len,(tr,va,te)))==(350,75,75)
assert not(set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))
split=np.full(len(X),'test',dtype=object); split[tr]='train'; split[va]='validation'
buffer=io.StringIO(newline=''); writer=csv.writer(buffer,lineterminator='\n')
writer.writerow(['row_id','split','x0','x1','x2','y'])
for i in ids: writer.writerow([int(i),split[i],*[format(v,'.17g') for v in X[i]],format(y[i],'.17g')])
DATA_BYTES=buffer.getvalue().encode(); DATA_HASH=hashlib.sha256(DATA_BYTES).hexdigest()
(ARTIFACTS/'frozen-data.csv').write_bytes(DATA_BYTES)
models=candidates(); validation={}
for name,model in models.items():
    model.fit(X[tr],y[tr]); validation[name]=metrics(y[va],model.predict(X[va]))
    transformed=add_square(X[tr]) if name=='quadratic' else X[tr]
    np.testing.assert_allclose(model.named_steps['standardscaler'].mean_,transformed.mean(axis=0))
selected_on_validation=min(['linear','quadratic'],key=lambda name:(validation[name]['MAE'],name!='linear'))
frozen={name:snapshot(model) for name,model in models.items()}
print('Split:',len(tr),len(va),len(te),'CSV SHA256:',DATA_HASH)
print('Validation only:',validation,'Saved choice:',selected_on_validation)

In [ ]:
responses["task3_explanation"] = ''

## Task 4: Diagnose both candidates (15 min)

**Purpose:** Connect a missing feature with actual error patterns and group scores.

**Do:** Open the predeclared test comparison without changing selected_on_validation. Plot prediction minus observation against x1 with shared x and y limits. Then inspect a separately labeled zoom. Report counts and MAE for abs(x1)<=1 and abs(x1)>1 for BOTH models; keep the boundary fixed. A known synthetic generator supports a representation explanation, not a physical causal claim.

**Check / stopping condition:** Both full-scale panels share identical limits. Both models use the same 75 test IDs and same slice membership. A zoom is labeled as a different scale.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task4_prediction"] = ''

In [ ]:
predictions={name:model.predict(X[te]) for name,model in models.items()}
residuals={name:pred-y[te] for name,pred in predictions.items()}
test_metrics={name:metrics(y[te],pred) for name,pred in predictions.items()}
mask=np.abs(X[te,1])>CONFIG['slice_boundary']; group_metrics={}
for name,pred in predictions.items():
    group_metrics[name]={}
    for label,m in [('inside',~mask),('outside',mask)]:
        assert m.any(); group_metrics[name][label]={'n':int(m.sum()),**metrics(y[te][m],pred[m])}
    assert sum(v['n'] for v in group_metrics[name].values())==75
print('Predeclared test comparison:',test_metrics,'Groups:',group_metrics)
fig,axes=plt.subplots(1,2,sharex=True,sharey=True,figsize=(10,4.5))
limit=1.1*max(np.max(np.abs(r)) for r in residuals.values())
for ax,(name,residual) in zip(axes, residuals.items()):
    ax.scatter(X[te,1],residual,c=COLORS[name],s=25); ax.axhline(0,c='black',lw=1)
    ax.set(title=name,xlabel='x1 (dimensionless)',ylim=(-limit,limit)); ax.grid(alpha=.2)
axes[0].set_ylabel('Prediction - observation (response units)')
fig.suptitle('Actual frozen candidates: same rows, shared horizontal and vertical scales')
savefig('04-shared-residuals.png')
plt.figure(); plt.scatter(X[te,1],residuals['quadratic'],c=COLORS['quadratic']); plt.axhline(0,c='black')
plt.xlabel('x1'); plt.ylabel('Prediction - observation'); plt.title('Quadratic detail ONLY: zoomed y axis, not the shared-scale comparison')
savefig('04-zoom.png')
plt.figure(); labels=['Inside |x1| <= 1','Outside |x1| > 1']; positions=np.arange(2)
for k,name in enumerate(models):
    vals=[group_metrics[name][g]['MAE'] for g in ['inside','outside']]
    plt.bar(positions+(k-.5)*.32,vals,.32,label=name,color=COLORS[name])
plt.xticks(positions,labels); plt.ylabel('MAE (response units)'); plt.title('Same fixed groups; report counts from the printed table'); plt.legend()
savefig('04-slice-errors.png')

In [ ]:
responses["task4_explanation"] = ''

## Task 5: Change one input distribution (12 min)

**Purpose:** Separate an input-distribution signal from a measured prediction error.

**Do:** Use 150 paired stress rows with a separate seed, not new training. Scale only x1 by 1, 1.5 and 2.5, preserving row pairing, x0, x2, noise and fitted parameters. Recompute targets using the SAME rule. Read the input histogram separately from labeled MAE. Do not assume every drift must harm every model.

**Check / stopping condition:** Three declared scales, two frozen-model error curves, and exact before/after parameter equality. Your measured values need not match another software version bit-for-bit.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task5_prediction"] = ''

In [ ]:
stress_rng=np.random.default_rng(CONFIG['seed']+100)
base=stress_rng.normal(size=(CONFIG['stress_n'],3)); stress_noise=stress_rng.normal(0,CONFIG['noise_sd'],len(base))
stress=[]
for scale in CONFIG['stress_scales']:
    shifted=base.copy(); shifted[:,1]*=scale
    target=2*shifted[:,0]+shifted[:,1]**2+stress_noise
    for name,model in models.items():
        stress.append({'scale':scale,'model':name,**metrics(target,model.predict(shifted))})
assert all(unchanged(frozen[name],model) for name,model in models.items())
print('Paired input-shift sweep:',stress)
fig,axes=plt.subplots(1,2,figsize=(10,4.5))
bins=np.linspace(-10,10,35)
for scale in [1.,2.5]: axes[0].hist(base[:,1]*scale,bins=bins,alpha=.5,label=f'x1 scale {scale}')
axes[0].set(xlabel='x1',ylabel='Count',title='Input signal; not accuracy'); axes[0].legend()
for name in models:
    rows=[r for r in stress if r['model']==name]
    axes[1].plot([r['scale'] for r in rows],[r['MAE'] for r in rows],'o-',label=name,color=COLORS[name])
axes[1].set(xlabel='Declared x1 scale',ylabel='MAE',title='Measured labeled performance'); axes[1].legend()
savefig('05-input-shift.png')

In [ ]:
responses["task5_explanation"] = ''

## Task 6: Change the relationship, keep inputs (15 min)

**Purpose:** Explain why an unchanged input monitor cannot certify current accuracy.

**Do:** Keep the exact test X, its order, existing predictions and original noise. Change ONLY the target rule from 2*x0+x1 squared+noise to -2*x0+x1 squared+noise. Verify the input hash is unchanged. Compare the original and changed-rule errors for both models. This is a constructed concept change, not evidence that real systems must behave this way.

**Check / stopping condition:** Identical before/after input hashes and frozen predictions. Labeled errors are recomputed against the new outcomes.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task6_prediction"] = ''

In [ ]:
input_hash_before=hashlib.sha256(X[te].tobytes()).hexdigest()
y_concept=-2*X[te,0]+X[te,1]**2+noise[te]
concept={name:metrics(y_concept,pred) for name,pred in predictions.items()}
input_hash_after=hashlib.sha256(X[te].tobytes()).hexdigest()
assert input_hash_before==input_hash_after
for name,model in models.items():
    assert unchanged(frozen[name],model)
    np.testing.assert_array_equal(model.predict(X[te]),predictions[name])
print('Input hashes identical:',input_hash_before==input_hash_after,'Changed-rule errors:',concept)
fig,axes=plt.subplots(1,2,figsize=(10,4.5))
axes[0].scatter(y[te],y_concept,c='#147c80',s=25)
limits=[min(y[te].min(),y_concept.min()),max(y[te].max(),y_concept.max())]
axes[0].plot(limits,limits,'--',c='gray'); axes[0].set(xlabel='Original outcome',ylabel='Changed-rule outcome',title='Same X, different outcome rule')
for k,name in enumerate(models):
    axes[1].bar(np.arange(2)+(k-.5)*.3,[test_metrics[name]['MAE'],concept[name]['MAE']],.3,label=name,color=COLORS[name])
axes[1].set_xticks([0,1],['Original rule','Changed rule']); axes[1].set(ylabel='MAE',title='Frozen predictions, new labels'); axes[1].legend()
savefig('06-concept-shift.png')

In [ ]:
responses["task6_explanation"] = ''

## Task 7: Check a one-coordinate explanation (12 min)

**Purpose:** Explain a fitted function without claiming causality or new accuracy.

**Do:** Hold every coordinate except x0 fixed. Read the raw-coordinate slope from ridge coefficient divided by scaler scale. Predict the change when x0 increases by 0.5, then compare the predicted delta with every observed prediction delta. No target labels are used in this check. It tests this function, not a physical intervention.

**Check / stopping condition:** All 75 prediction changes match 0.5 times the raw x0 slope, within the declared numerical tolerance. Standardized coefficients alone are not original-unit slopes.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task7_prediction"] = ''

In [ ]:
perturbed=X[te].copy(); perturbed[:,0]+=CONFIG['delta_x0']; perturb={}; deltas={}
for name,model in models.items():
    slope=float(model.named_steps['ridge'].coef_[0]/model.named_steps['standardscaler'].scale_[0])
    delta=model.predict(perturbed)-predictions[name]; expected=CONFIG['delta_x0']*slope
    np.testing.assert_allclose(delta,np.full(len(te),expected),rtol=CONFIG['prediction_rtol'],atol=CONFIG['prediction_atol'])
    deltas[name]=delta; perturb[name]={'raw_x0_slope':slope,'expected_delta':expected,
          'max_disagreement':float(np.max(np.abs(delta-expected))),'rows_checked':len(te)}
print('Function check:',perturb)
plt.figure()
for name,delta in deltas.items(): plt.plot(ids[te],delta,'.',label=name,color=COLORS[name])
plt.xlabel('Original test row ID'); plt.ylabel('Prediction change'); plt.title('Add 0.5 to x0 only: compare every row'); plt.legend(); savefig('07-function-check.png')

In [ ]:
responses["task7_explanation"] = ''

## Task 8: Read back, align and export (13 min)

**Purpose:** Preserve data identity and honestly state replay scope.

**Do:** Read the saved CSV bytes, check their hash, sort by stable row ID and reconstruct split membership. Clone the already-selected candidate using code/configuration from this kernel, refit on CSV training rows and compare ALL aligned test predictions. This is same-kernel CSV readback consistency, not fresh-environment reproduction. Finish the three reflection fields and export below; leave a draft if unfinished.

**Check / stopping condition:** 500 unique IDs; identical splits; aligned test predictions agree within declared tolerance. Hash means matching bytes against a reference, not sender authenticity or scientific truth.

First fill the prediction cell below, then run the scaffold. Finally fill the explanation cell with one observed result and its boundary.

In [ ]:
responses["task8_prediction"] = ''

In [ ]:
read_bytes=(ARTIFACTS/'frozen-data.csv').read_bytes(); assert hashlib.sha256(read_bytes).hexdigest()==DATA_HASH
records=list(csv.DictReader(io.StringIO(read_bytes.decode())))
records.sort(key=lambda r:int(r['row_id']))
read_ids=np.array([int(r['row_id']) for r in records]); np.testing.assert_array_equal(read_ids,ids)
assert len(set(read_ids))==CONFIG['n']
read_X=np.array([[float(r[k]) for k in ['x0','x1','x2']] for r in records]); read_y=np.array([float(r['y']) for r in records])
read_split=np.array([r['split'] for r in records]); np.testing.assert_array_equal(read_split,split)
read_tr=np.flatnonzero(read_split=='train'); read_te=np.flatnonzero(read_split=='test')
np.testing.assert_array_equal(read_te,te)
replayed=clone(models[selected_on_validation]); replayed.fit(read_X[read_tr],read_y[read_tr])
read_pred=replayed.predict(read_X[read_te]); reference_pred=predictions[selected_on_validation]
np.testing.assert_allclose(read_pred,reference_pred,rtol=CONFIG['prediction_rtol'],atol=CONFIG['prediction_atol'])
readback={'scope':'same-kernel CSV readback consistency; not independent-process reproduction',
          'data_sha256':DATA_HASH,'rows':len(read_ids),'test_rows':len(read_te),
          'max_abs_prediction_difference':float(np.max(np.abs(read_pred-reference_pred))),
          'selected_on_validation':selected_on_validation}
assert all(unchanged(frozen[name],model) for name,model in models.items())
print(readback)
plt.figure(); plt.plot(read_te,read_pred-reference_pred,'o',c='#147c80'); plt.axhline(0,c='black')
plt.xlabel('Aligned test row ID'); plt.ylabel('Readback prediction - reference'); plt.ylim(-1e-9,1e-9)
plt.title('Same-kernel CSV readback: zero-centered difference, fixed displayed scale'); savefig('08-readback.png')

In [ ]:
responses["task8_explanation"] = ''

## Optional A and B

Core work ends at Task 8. Optional A holds labels fixed while introducing and decoding a dimensionless x0 encoding error. Optional B resamples a fixed model’s errors within the predeclared groups (300 draws), assuming IID cases within each group. Its percentile interval excludes training uncertainty and arbitrary future shift. Predict the distinction before setting `RUN_CHALLENGES=True`. Both branches leave core data and fitted models unchanged.

In [ ]:
RUN_CHALLENGES=False  # Change to True only after the eight core tasks.
if RUN_CHALLENGES:
    # A: an encoding mistake, NOT a physical unit conversion.
    encoded=X[te].copy(); encoded[:,0]*=100
    decoded=encoded.copy(); decoded[:,0]/=100
    optional_encoding={}
    for name,model in models.items():
        np.testing.assert_allclose(model.predict(decoded),predictions[name],atol=1e-10,rtol=1e-10)
        optional_encoding[name]={'wrong_encoding':metrics(y[te],model.predict(encoded)),
                                 'decoded':metrics(y[te],model.predict(decoded))}
    print('Optional A: same labels, dimensionless x0 encoding error:',optional_encoding)
    # B: conditional on a fixed model; assumes IID cases within each selected slice.
    bootstrap_rng=np.random.default_rng(CONFIG['seed']+200)
    optional_bootstrap={}
    for label,m in [('inside',~mask),('outside',mask)]:
        errors=np.abs(residuals[selected_on_validation][m])
        draws=bootstrap_rng.choice(errors,size=(300,len(errors)),replace=True).mean(axis=1)
        optional_bootstrap[label]={'n':len(errors),'percentile_interval':np.quantile(draws,[.025,.975]).tolist()}
    print('Optional B: fixed-model within-slice bootstrap; excludes training variability and arbitrary future shift:',optional_bootstrap)
    (ARTIFACTS/'optional-results.json').write_text(json.dumps({'encoding':optional_encoding,'bootstrap':optional_bootstrap},indent=2))

## Reflect, peer-review and download

Use the remaining 20 studio minutes to show a partner one actual failure, the matching plot, what changed, what stayed fixed and one limitation. Complete the three fields below. Write `none` for AI assistance only if that is accurate. Eight predictions + eight explanations + three reflections = **19 fields**. Blanks produce an honest draft; nonempty fields only mean ready for peer discussion, not automatically correct. The supplied source file is the original scaffold; save your edited notebook separately if you change code.

In [ ]:
responses['controlled_change'] = ''
responses['limitation'] = ''
responses['ai_disclosure'] = ''

In [ ]:
# @title Preserve supplied scaffold source (run; no edits required)
ANALYSIS_SOURCE="\nfrom pathlib import Path\nimport sys, subprocess, importlib.util\nneeded={'numpy':'numpy','matplotlib':'matplotlib','sklearn':'scikit-learn'}\nmissing=[package for module,package in needed.items() if importlib.util.find_spec(module) is None]\nif missing:\n    subprocess.check_call([sys.executable,'-m','pip','install',*missing])\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport sklearn, json, hashlib, csv, io, zipfile, platform\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler, FunctionTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.linear_model import Ridge\nfrom sklearn.base import clone\nfrom IPython.display import display, FileLink\nARTIFACTS=Path('week09-artifacts'); ARTIFACTS.mkdir(exist_ok=True)\n# Clear old outputs from this notebook's own output folder, never other folders.\nfor old in ARTIFACTS.iterdir():\n    if old.is_file(): old.unlink()\nCONFIG={'seed':2026,'n':500,'noise_sd':0.2,'ridge_alpha':1.0,\n        'stress_scales':[1.0,1.5,2.5],'stress_n':150,'slice_boundary':1.0,\n        'delta_x0':0.5,'prediction_atol':1e-10,'prediction_rtol':1e-10}\nresponses={}\nVERSIONS={'python':platform.python_version(),'numpy':np.__version__,\n          'scikit-learn':sklearn.__version__}\nplt.rcParams.update({'figure.figsize':(9,4.8),'font.size':12,'axes.spines.top':False,\n                     'axes.spines.right':False,'figure.dpi':120})\nCOLORS={'linear':'#a43a4b','quadratic':'#147c80'}\ndef metrics(y_true,pred):\n    residual=np.asarray(pred)-np.asarray(y_true)\n    return {'MAE':float(np.mean(np.abs(residual))), 'RMSE':float(np.sqrt(np.mean(residual**2)))}\ndef savefig(name):\n    plt.tight_layout(); plt.savefig(ARTIFACTS/name,bbox_inches='tight'); plt.show(); plt.close()\ndef add_square(a):\n    a=np.asarray(a); return np.column_stack([a,a[:,1]**2])\ndef candidates():\n    return {'linear':make_pipeline(StandardScaler(),Ridge(alpha=CONFIG['ridge_alpha'],solver='svd')),\n            'quadratic':make_pipeline(FunctionTransformer(add_square,validate=True),StandardScaler(),\n                                      Ridge(alpha=CONFIG['ridge_alpha'],solver='svd'))}\ndef snapshot(model):\n    scaler=model.named_steps['standardscaler']; ridge=model.named_steps['ridge']\n    return [a.copy() for a in [scaler.mean_,scaler.scale_,ridge.coef_,np.atleast_1d(ridge.intercept_)]]\ndef unchanged(before,model):\n    return all(np.array_equal(a,b) for a,b in zip(before,snapshot(model)))\nprint('CPU environment:',VERSIONS)\nprint('No API key, GPU, external dataset or paid AI tool is needed.')\n\n\nx_demo=np.array([-2.,-1.,0.,1.,2.]); y_demo=x_demo**2\npred_demo=np.ones(5); residual_demo=pred_demo-y_demo\nassert np.array_equal(residual_demo,[-3,0,1,0,-3])\nwarmup=metrics(y_demo,pred_demo)\nprint('x1 | observed | predicted | residual')\nfor row in zip(x_demo,y_demo,pred_demo,residual_demo): print(row)\nprint('Absolute sum:',np.abs(residual_demo).sum(),'Squared sum:',(residual_demo**2).sum(),warmup)\nfig,axes=plt.subplots(1,2,figsize=(10,4))\naxes[0].scatter(x_demo,y_demo,label='Observed square',c='#147c80',s=70)\naxes[0].plot(x_demo,pred_demo,'o--',label='Fixed prediction = 1',c='#a43a4b')\naxes[0].vlines(x_demo,np.minimum(y_demo,pred_demo),np.maximum(y_demo,pred_demo),color='#64748b')\naxes[0].set(xlabel='x1 (dimensionless)',ylabel='Synthetic response'); axes[0].legend()\naxes[1].axhline(0,color='black'); axes[1].bar(x_demo,residual_demo,color=['#a43a4b' if x<0 else '#147c80' for x in residual_demo])\naxes[1].set(xlabel='x1',ylabel='Prediction - observation',ylim=(-4,2))\nfig.suptitle('Teaching slice: a declared approximation, not a fitted experiment')\nsavefig('01-five-errors.png')\n\n\nmixture=[]\nfor easy,hard in [(90,10),(75,25)]:\n    parts=[easy*1/100,hard*10/100]\n    mixture.append({'easy':easy,'hard':hard,'parts':parts,'MAE':sum(parts)})\nprint(mixture)\nassert np.allclose([a['MAE'] for a in mixture],[1.9,3.25])\nplt.figure(); bottoms=np.array([r['parts'][0] for r in mixture]); tops=np.array([r['parts'][1] for r in mixture])\nplt.bar(['90 / 10','75 / 25'],bottoms,label='Easy contribution',color='#147c80')\nplt.bar(['90 / 10','75 / 25'],tops,bottom=bottoms,label='Difficult contribution',color='#a43a4b')\nplt.ylabel('Weighted MAE (synthetic response units)'); plt.xlabel('Easy / difficult case counts (100 total)')\nplt.title('Paper illustration: same group losses, changed mixture'); plt.legend(); savefig('02-mixture.png')\n\n\nrng=np.random.default_rng(CONFIG['seed']); X=rng.normal(size=(CONFIG['n'],3))\nnoise=rng.normal(0,CONFIG['noise_sd'],size=CONFIG['n']); y=2*X[:,0]+X[:,1]**2+noise\nids=np.arange(len(X)); tr,held=train_test_split(ids,test_size=.30,random_state=CONFIG['seed'])\nva,te=train_test_split(held,test_size=.50,random_state=CONFIG['seed']+1)\ntr,va,te=map(np.sort,(tr,va,te)); assert tuple(map(len,(tr,va,te)))==(350,75,75)\nassert not(set(tr)&set(va) or set(tr)&set(te) or set(va)&set(te))\nsplit=np.full(len(X),'test',dtype=object); split[tr]='train'; split[va]='validation'\nbuffer=io.StringIO(newline=''); writer=csv.writer(buffer,lineterminator='\\n')\nwriter.writerow(['row_id','split','x0','x1','x2','y'])\nfor i in ids: writer.writerow([int(i),split[i],*[format(v,'.17g') for v in X[i]],format(y[i],'.17g')])\nDATA_BYTES=buffer.getvalue().encode(); DATA_HASH=hashlib.sha256(DATA_BYTES).hexdigest()\n(ARTIFACTS/'frozen-data.csv').write_bytes(DATA_BYTES)\nmodels=candidates(); validation={}\nfor name,model in models.items():\n    model.fit(X[tr],y[tr]); validation[name]=metrics(y[va],model.predict(X[va]))\n    transformed=add_square(X[tr]) if name=='quadratic' else X[tr]\n    np.testing.assert_allclose(model.named_steps['standardscaler'].mean_,transformed.mean(axis=0))\nselected_on_validation=min(['linear','quadratic'],key=lambda name:(validation[name]['MAE'],name!='linear'))\nfrozen={name:snapshot(model) for name,model in models.items()}\nprint('Split:',len(tr),len(va),len(te),'CSV SHA256:',DATA_HASH)\nprint('Validation only:',validation,'Saved choice:',selected_on_validation)\n\n\npredictions={name:model.predict(X[te]) for name,model in models.items()}\nresiduals={name:pred-y[te] for name,pred in predictions.items()}\ntest_metrics={name:metrics(y[te],pred) for name,pred in predictions.items()}\nmask=np.abs(X[te,1])>CONFIG['slice_boundary']; group_metrics={}\nfor name,pred in predictions.items():\n    group_metrics[name]={}\n    for label,m in [('inside',~mask),('outside',mask)]:\n        assert m.any(); group_metrics[name][label]={'n':int(m.sum()),**metrics(y[te][m],pred[m])}\n    assert sum(v['n'] for v in group_metrics[name].values())==75\nprint('Predeclared test comparison:',test_metrics,'Groups:',group_metrics)\nfig,axes=plt.subplots(1,2,sharex=True,sharey=True,figsize=(10,4.5))\nlimit=1.1*max(np.max(np.abs(r)) for r in residuals.values())\nfor ax,(name,residual) in zip(axes, residuals.items()):\n    ax.scatter(X[te,1],residual,c=COLORS[name],s=25); ax.axhline(0,c='black',lw=1)\n    ax.set(title=name,xlabel='x1 (dimensionless)',ylim=(-limit,limit)); ax.grid(alpha=.2)\naxes[0].set_ylabel('Prediction - observation (response units)')\nfig.suptitle('Actual frozen candidates: same rows, shared horizontal and vertical scales')\nsavefig('04-shared-residuals.png')\nplt.figure(); plt.scatter(X[te,1],residuals['quadratic'],c=COLORS['quadratic']); plt.axhline(0,c='black')\nplt.xlabel('x1'); plt.ylabel('Prediction - observation'); plt.title('Quadratic detail ONLY: zoomed y axis, not the shared-scale comparison')\nsavefig('04-zoom.png')\nplt.figure(); labels=['Inside |x1| <= 1','Outside |x1| > 1']; positions=np.arange(2)\nfor k,name in enumerate(models):\n    vals=[group_metrics[name][g]['MAE'] for g in ['inside','outside']]\n    plt.bar(positions+(k-.5)*.32,vals,.32,label=name,color=COLORS[name])\nplt.xticks(positions,labels); plt.ylabel('MAE (response units)'); plt.title('Same fixed groups; report counts from the printed table'); plt.legend()\nsavefig('04-slice-errors.png')\n\n\nstress_rng=np.random.default_rng(CONFIG['seed']+100)\nbase=stress_rng.normal(size=(CONFIG['stress_n'],3)); stress_noise=stress_rng.normal(0,CONFIG['noise_sd'],len(base))\nstress=[]\nfor scale in CONFIG['stress_scales']:\n    shifted=base.copy(); shifted[:,1]*=scale\n    target=2*shifted[:,0]+shifted[:,1]**2+stress_noise\n    for name,model in models.items():\n        stress.append({'scale':scale,'model':name,**metrics(target,model.predict(shifted))})\nassert all(unchanged(frozen[name],model) for name,model in models.items())\nprint('Paired input-shift sweep:',stress)\nfig,axes=plt.subplots(1,2,figsize=(10,4.5))\nbins=np.linspace(-10,10,35)\nfor scale in [1.,2.5]: axes[0].hist(base[:,1]*scale,bins=bins,alpha=.5,label=f'x1 scale {scale}')\naxes[0].set(xlabel='x1',ylabel='Count',title='Input signal; not accuracy'); axes[0].legend()\nfor name in models:\n    rows=[r for r in stress if r['model']==name]\n    axes[1].plot([r['scale'] for r in rows],[r['MAE'] for r in rows],'o-',label=name,color=COLORS[name])\naxes[1].set(xlabel='Declared x1 scale',ylabel='MAE',title='Measured labeled performance'); axes[1].legend()\nsavefig('05-input-shift.png')\n\n\ninput_hash_before=hashlib.sha256(X[te].tobytes()).hexdigest()\ny_concept=-2*X[te,0]+X[te,1]**2+noise[te]\nconcept={name:metrics(y_concept,pred) for name,pred in predictions.items()}\ninput_hash_after=hashlib.sha256(X[te].tobytes()).hexdigest()\nassert input_hash_before==input_hash_after\nfor name,model in models.items():\n    assert unchanged(frozen[name],model)\n    np.testing.assert_array_equal(model.predict(X[te]),predictions[name])\nprint('Input hashes identical:',input_hash_before==input_hash_after,'Changed-rule errors:',concept)\nfig,axes=plt.subplots(1,2,figsize=(10,4.5))\naxes[0].scatter(y[te],y_concept,c='#147c80',s=25)\nlimits=[min(y[te].min(),y_concept.min()),max(y[te].max(),y_concept.max())]\naxes[0].plot(limits,limits,'--',c='gray'); axes[0].set(xlabel='Original outcome',ylabel='Changed-rule outcome',title='Same X, different outcome rule')\nfor k,name in enumerate(models):\n    axes[1].bar(np.arange(2)+(k-.5)*.3,[test_metrics[name]['MAE'],concept[name]['MAE']],.3,label=name,color=COLORS[name])\naxes[1].set_xticks([0,1],['Original rule','Changed rule']); axes[1].set(ylabel='MAE',title='Frozen predictions, new labels'); axes[1].legend()\nsavefig('06-concept-shift.png')\n\n\nperturbed=X[te].copy(); perturbed[:,0]+=CONFIG['delta_x0']; perturb={}; deltas={}\nfor name,model in models.items():\n    slope=float(model.named_steps['ridge'].coef_[0]/model.named_steps['standardscaler'].scale_[0])\n    delta=model.predict(perturbed)-predictions[name]; expected=CONFIG['delta_x0']*slope\n    np.testing.assert_allclose(delta,np.full(len(te),expected),rtol=CONFIG['prediction_rtol'],atol=CONFIG['prediction_atol'])\n    deltas[name]=delta; perturb[name]={'raw_x0_slope':slope,'expected_delta':expected,\n          'max_disagreement':float(np.max(np.abs(delta-expected))),'rows_checked':len(te)}\nprint('Function check:',perturb)\nplt.figure()\nfor name,delta in deltas.items(): plt.plot(ids[te],delta,'.',label=name,color=COLORS[name])\nplt.xlabel('Original test row ID'); plt.ylabel('Prediction change'); plt.title('Add 0.5 to x0 only: compare every row'); plt.legend(); savefig('07-function-check.png')\n\n\nread_bytes=(ARTIFACTS/'frozen-data.csv').read_bytes(); assert hashlib.sha256(read_bytes).hexdigest()==DATA_HASH\nrecords=list(csv.DictReader(io.StringIO(read_bytes.decode())))\nrecords.sort(key=lambda r:int(r['row_id']))\nread_ids=np.array([int(r['row_id']) for r in records]); np.testing.assert_array_equal(read_ids,ids)\nassert len(set(read_ids))==CONFIG['n']\nread_X=np.array([[float(r[k]) for k in ['x0','x1','x2']] for r in records]); read_y=np.array([float(r['y']) for r in records])\nread_split=np.array([r['split'] for r in records]); np.testing.assert_array_equal(read_split,split)\nread_tr=np.flatnonzero(read_split=='train'); read_te=np.flatnonzero(read_split=='test')\nnp.testing.assert_array_equal(read_te,te)\nreplayed=clone(models[selected_on_validation]); replayed.fit(read_X[read_tr],read_y[read_tr])\nread_pred=replayed.predict(read_X[read_te]); reference_pred=predictions[selected_on_validation]\nnp.testing.assert_allclose(read_pred,reference_pred,rtol=CONFIG['prediction_rtol'],atol=CONFIG['prediction_atol'])\nreadback={'scope':'same-kernel CSV readback consistency; not independent-process reproduction',\n          'data_sha256':DATA_HASH,'rows':len(read_ids),'test_rows':len(read_te),\n          'max_abs_prediction_difference':float(np.max(np.abs(read_pred-reference_pred))),\n          'selected_on_validation':selected_on_validation}\nassert all(unchanged(frozen[name],model) for name,model in models.items())\nprint(readback)\nplt.figure(); plt.plot(read_te,read_pred-reference_pred,'o',c='#147c80'); plt.axhline(0,c='black')\nplt.xlabel('Aligned test row ID'); plt.ylabel('Readback prediction - reference'); plt.ylim(-1e-9,1e-9)\nplt.title('Same-kernel CSV readback: zero-centered difference, fixed displayed scale'); savefig('08-readback.png')\n\n\nRUN_CHALLENGES=False  # Change to True only after the eight core tasks.\nif RUN_CHALLENGES:\n    # A: an encoding mistake, NOT a physical unit conversion.\n    encoded=X[te].copy(); encoded[:,0]*=100\n    decoded=encoded.copy(); decoded[:,0]/=100\n    optional_encoding={}\n    for name,model in models.items():\n        np.testing.assert_allclose(model.predict(decoded),predictions[name],atol=1e-10,rtol=1e-10)\n        optional_encoding[name]={'wrong_encoding':metrics(y[te],model.predict(encoded)),\n                                 'decoded':metrics(y[te],model.predict(decoded))}\n    print('Optional A: same labels, dimensionless x0 encoding error:',optional_encoding)\n    # B: conditional on a fixed model; assumes IID cases within each selected slice.\n    bootstrap_rng=np.random.default_rng(CONFIG['seed']+200)\n    optional_bootstrap={}\n    for label,m in [('inside',~mask),('outside',mask)]:\n        errors=np.abs(residuals[selected_on_validation][m])\n        draws=bootstrap_rng.choice(errors,size=(300,len(errors)),replace=True).mean(axis=1)\n        optional_bootstrap[label]={'n':len(errors),'percentile_interval':np.quantile(draws,[.025,.975]).tolist()}\n    print('Optional B: fixed-model within-slice bootstrap; excludes training variability and arbitrary future shift:',optional_bootstrap)\n    (ARTIFACTS/'optional-results.json').write_text(json.dumps({'encoding':optional_encoding,'bootstrap':optional_bootstrap},indent=2))\n"

In [ ]:
required=[f'task{i}_{kind}' for i in range(1,9) for kind in ['prediction','explanation']]+['controlled_change','limitation','ai_disclosure']
assert len(required)==19
blank=[k for k in required if not isinstance(responses.get(k),str) or not responses[k].strip()]
status='draft' if blank else 'ready for peer discussion; completeness only, not correctness'
results={'warmup':warmup,'mixture':mixture,'validation':validation,'selected_on_validation':selected_on_validation,
         'test':test_metrics,'groups':group_metrics,'stress':stress,'concept':concept,'perturb':perturb,'readback':readback}
(ARTIFACTS/'results.json').write_text(json.dumps(results,indent=2))
(ARTIFACTS/'config.json').write_text(json.dumps(CONFIG,indent=2))
(ARTIFACTS/'versions.json').write_text(json.dumps(VERSIONS,indent=2))
(ARTIFACTS/'responses.json').write_text(json.dumps({'status':status,'blank_fields':blank,'responses':responses},indent=2))
# Save predictions with IDs so a reviewer can compare alignment, not just a scalar mean.
with (ARTIFACTS/'test-predictions.csv').open('w',newline='') as f:
    writer=csv.writer(f); writer.writerow(['row_id','observed','linear','quadratic'])
    for j,i in enumerate(te): writer.writerow([int(i),y[i],predictions['linear'][j],predictions['quadratic'][j]])
(ARTIFACTS/'supplied-analysis-source.py').write_text(ANALYSIS_SOURCE)
files={p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(ARTIFACTS.iterdir()) if p.is_file() and p.name!='manifest.json'}
manifest={'week':9,'status':status,'response_fields':len(required),'completed_fields':19-len(blank),'files':files,
          'scope':readback['scope'],'note':'Save your edited notebook separately; the supplied source is a scaffold snapshot.'}
(ARTIFACTS/'manifest.json').write_text(json.dumps(manifest,indent=2))
zip_path=Path('week09-evidence.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for p in sorted(ARTIFACTS.iterdir()):
        if p.is_file(): archive.write(p,arcname=p.name)
print('Export:',status,'Completed response fields:',19-len(blank),'/ 19','ZIP bytes:',zip_path.stat().st_size)
print('Blank fields:',blank,'Save your edited notebook and download the ZIP before the runtime ends.')
display(FileLink(str(zip_path)))
try:
    from google.colab import files as colab_files
    # Explicit student click: uncomment only when ready to download.
    # colab_files.download(str(zip_path))
except ImportError:
    pass

## Summary and next step

Locate weak cases before trusting an average. Connect a residual pattern to the missing representation. Distinguish changing inputs from a changed target rule. Check an explanation against the fitted function. Preserve data, row alignment, results and limitations. Week 10 extends the same-kernel readback to a more portable workflow.

## References

- [Monitoring ML pipelines](https://developers.google.com/machine-learning/crash-course/production-ml-systems/monitoring)
- [Leakage-safe pipelines](https://scikit-learn.org/stable/common_pitfalls.html)
- [Regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)

Original synthetic examples and teaching scaffolds. Source documentation was checked on 27 October 2026.